In [ ]:
%load_ext autoreload
%autoreload 2

# pywatemsedem modelinput and modeloutput

## 1. Introduction

This tutorial describes the use of Modelinput and Modeloutput class of the pywatemsedem Python package. Via these classes the scenario input and output can be dynamically loaded into memory. The Python package also contains a Postprocessing class which makes use of these loaded inputs and outputs, yet this is not handled in this notebook.

## 2. Imports and example data

In [ ]:
import os
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

In [ ]:
# Go to project root
os.chdir("../..")

# Example results folder
results_folder = Path("tests/io/data")

## 3. Modelinput and Modeloutput objects

### 3.1. Initialisation

The object initialisation arguments can be fetched based on ini file and the epsg code

In [ ]:
ini = results_folder / "modelinput/inifile.ini"
epsg = 31370

from pywatemsedem.geo.utils import get_rstparams
_, rp = get_rstparams(ini, epsg=epsg)
resolution = rp["res"]
nodata = rp["nodata"]

The Modelinput and Modeloutput objects can be initialised with the following commands

In [ ]:
from pywatemsedem.io.modelinput import Modelinput
modelinput = Modelinput(ini, resolution, epsg, nodata)

from pywatemsedem.io.modeloutput import Modeloutput
modeloutput = Modeloutput(ini, resolution, epsg, nodata)

### 3.2. Map variables

Load the mask and rivermask variable

In [ ]:
mask = modelinput.mask
rivermask = modelinput.rivermask

In [ ]:
rivermask.plot()

Load a map variable: 
- Input variables: 

    "mask","rivermask", "buffers", "cfactor", "dtm", "kfactor", "ktc", "outlet", "pfactor", "compositelanduse", "ptef", "riversegments", "riverrouting"

- Output variables:

    "aspect", "capacity", "cumulative", "ls", "rusle", "sedi_export", "sedi_in", "sedi_out", "slope", "uparea", "watereros_kg", "watereros_mm"

In [ ]:
obj = modeloutput # modelinput or modeloutput
variable_name = "sedi_out"
variable = getattr(obj, variable_name)

Plot of the selected variable

In [ ]:
variable.plot()

Check variable statistics for multiple grid cell selections:
- all grid cells
- catchment grid cells
- land grid cells (within catchment)
- river grid cells (within catchment)

In [ ]:
def get_stats(arr):
    return pd.Series(arr).describe().round(2)
pd.DataFrame({
    "all grid cells": get_stats(variable.arr.flatten()),
    "catchment grid cells": get_stats(variable.arr[mask.arr!=-9999]),
    "land grid cells": get_stats(variable.arr[(mask.arr!=-9999) & (rivermask.arr==-9999)]),
    "river grid cells": get_stats(variable.arr[rivermask.arr!=-9999]),
})

### 3.3 Table variables

Load a table variable: 
- Input variables: 

    "adjacent_segments", "upstream_segments"

- Output variables:

    "cumulative_sediment_segments", "routing_missing", "routing", "total_sediment_segments", "total_sediment", 

In [ ]:
obj = modeloutput # modelinput or modeloutput
variable_name = "routing"
variable = getattr(obj, variable_name)

In [ ]:
variable

In [ ]:
from pywatemsedem.io.modeloutput import (
    Modeloutput,
    check_segment_edges,
    compute_efficiency_buffers,
    identify_rank_sediment_loads,
)
from pathlib import Path

In [ ]:
tag = "rank"
folder = Path(r"tests/io/data")

filepath_out = folder / "modeloutput" / "SediExport_kg.rst"

df_export, _ = identify_rank_sediment_loads(
    filepath_out,
    50,
    Path(os.getcwd()) / (tag + ".rst"),
    rst_endpoints=folder / "modeloutput" / "sewer_in.rst",
)
df_export = df_export[df_export["class"] != -9999]

In [ ]:
df_export